In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv('csgo.csv')
print(df.columns)
print(df.head(5))


Index(['map', 'day', 'month', 'year', 'date', 'wait_time_s', 'match_time_s',
       'team_a_rounds', 'team_b_rounds', 'ping', 'kills', 'assists', 'deaths',
       'mvps', 'hs_percent', 'points', 'result'],
      dtype='str')
      map   day  month    year       date  wait_time_s  match_time_s  \
0  Mirage   3.0    8.0  2018.0   3/8/2018        327.0        2906.0   
1  Mirage   2.0    8.0  2018.0   2/8/2018        336.0        2592.0   
2  Mirage  31.0    7.0  2018.0  31/7/2018        414.0        2731.0   
3  Mirage  31.0    7.0  2018.0  31/7/2018        317.0        2379.0   
4  Mirage  30.0    7.0  2018.0  30/7/2018        340.0        3467.0   

   team_a_rounds  team_b_rounds   ping  kills  assists  deaths  mvps  \
0           16.0           13.0  215.0   17.0      2.0    21.0   2.0   
1           16.0           11.0  199.0   13.0      4.0    24.0   2.0   
2           16.0           14.0   85.0   15.0      3.0    18.0   3.0   
3           11.0           16.0   93.0   12.0      2.0

In [3]:
df.drop(columns = ['day','month','year','date','wait_time_s','team_a_rounds','team_b_rounds'],inplace = True)


# ***Chia train-test***


In [13]:
target = 'result'
x = df.drop([target],axis = 1)
y = df[target]


In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 36, stratify = y
)

Encode cho feature 'Map':

In [15]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # Gán map = 0 trên test nếu có map lạ
map_train = encoder.fit_transform(x_train[['map']])
map_test = encoder.transform(x_test[['map']])

encoded_cols = encoder.get_feature_names_out(['map'])

# Chuyển thành df
df_map_train = pd.DataFrame(map_train, columns=encoded_cols, index = x_train.index)
df_map_test = pd.DataFrame(map_test, columns=encoded_cols, index = x_test.index)

x_train = pd.concat([x_train.drop('map', axis=1), df_map_train], axis=1)
x_test = pd.concat([x_test.drop('map', axis=1), df_map_test], axis=1)

# Train RandomForest

In [16]:
model = RandomForestClassifier(
    n_estimators = 100,
    random_state = 36,
    class_weight='balanced'
)
skf = StratifiedKFold(
    n_splits = 5,
    random_state = 36,
    shuffle = True
)
cvs = cross_val_score(
    model,
    x_train,
    y_train,
    cv = skf,
    scoring = 'f1_macro',
)
print('Avg f1 macro: ',cvs.mean())

Avg f1 macro:  0.626927058752872


In [17]:
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
print(model.score(x_test, y_test))
print(classification_report(y_test, y_pred))

0.7533039647577092
              precision    recall  f1-score   support

        Lost       0.76      0.80      0.78       112
         Tie       0.57      0.24      0.33        17
         Win       0.76      0.79      0.77        98

    accuracy                           0.75       227
   macro avg       0.70      0.61      0.63       227
weighted avg       0.75      0.75      0.74       227

